# 🧪 Benchmark d'Évaluation — Pipeline RAG Neurosymbolique (Briques 1→5)

**Objectif** : Évaluer la précision de notre pipeline IA sur les réponses réelles de **5 cardiologues** × **15 cas ECG**.

**Score STRICT** = moyenne des % des **diagnostics validants attendus** matchés (découvertes additionnelles trackées séparément, 0 pt).

**Pipeline testé** :
1. **Brique 2** — Extraction NER (GPT-4o Structured Outputs)
2. **Brique 3** — Recherche Hybride (Dense + BM25 + RRF)
3. **Brique 4** — Juge Neurosymbolique (Coupe-Circuit + GPT-4o-mini QCM)
4. **Brique 5** — Scoring ensembliste : attendus (∩) vs découvertes (—)

| Cellule | Section | Description |
|---------|---------|-------------|
| 1 | **Setup** | Imports + Initialisation moteur RAG + scoring.py |
| 2 | **Données** | Chargement CSV + Golden Set (validants / descripteurs) |
| 3 | **Pipeline** | Boucle de traitement (score strict = attendus validés uniquement) |
| 4 | **Audit** | Heatmap + Audit détaillé + Découvertes additionnelles |

In [10]:
# ============================================================
# CELLULE 1 — Imports & Initialisation du Moteur RAG
# ============================================================
import sys, os, json, time, warnings, importlib
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Chemins racines ──────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\Administrateur\bmad\ECG lecture")
EVAL_ROOT    = Path(r"C:\Users\Administrateur\bmad\ECG evaluation")
RAG_ROOT     = Path(r"C:\Users\Administrateur\bmad\RAG ontologique")

# Ajouter au PYTHONPATH pour les imports
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(RAG_ROOT))

# Charger la clé API
load_dotenv(PROJECT_ROOT / ".env")

# ─── Imports RAG Neurosymbolique (Briques 2, 3, 4) ───────────────────────────
# Force reload pour utiliser les versions modifiées sur disque
import ner_extractor; importlib.reload(ner_extractor)
import ontology_index; importlib.reload(ontology_index)
import hybrid_search; importlib.reload(hybrid_search)
import neurosymbolic_judge; importlib.reload(neurosymbolic_judge)

from ner_extractor import extract_clinical_terms, NERExtraction
from hybrid_search import HybridSearchEngine
from neurosymbolic_judge import resolve_term_to_ontology

# ─── Import du scoring de production (find_owl_concept + implications) ────────
import scoring; importlib.reload(scoring)
from scoring import find_owl_concept, apply_implication_rules, score_student_response
from scoring import SCORE_BY_GENERATION, SCORE_GENERATION_FLOOR

# ─── Initialiser le moteur de recherche hybride en RAM (une seule fois) ───────
os.chdir(str(RAG_ROOT))  # Pour que HybridSearchEngine trouve rag_index/
moteur = HybridSearchEngine()

# ─── Charger l'ontologie (source unique : ECG lecture/data/) ──────────────────
with open(PROJECT_ROOT / "data" / "ontology_from_owl.json", 'r', encoding='utf-8') as f:
    ONTOLOGY = json.load(f)

CONCEPT_MAPPINGS = ONTOLOGY.get('concept_mappings', {})
IMPLICATION_RULES = ONTOLOGY.get('implication_rules', {})

# ─── Vérifications ───────────────────────────────────────────────────────────
print(f"[OK] OPENAI_API_KEY : {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"[OK] HybridSearchEngine : {len(moteur.documents)} documents indexés")
print(f"[OK] Ontologie : {len(CONCEPT_MAPPINGS)} concepts chargés")
print(f"[OK] Implications : {len(IMPLICATION_RULES)} règles")
print(f"[OK] Barème dégressif : Gen1={SCORE_BY_GENERATION[1]:.0f}%, Gen2={SCORE_BY_GENERATION[2]:.0f}%, Gen3={SCORE_BY_GENERATION[3]:.0f}%, Gen4+={SCORE_GENERATION_FLOOR:.0f}%")

# Vérification clé : TV est-il indexé pour TACHYCARDIE_VENTRICULAIRE ?
tv_forms = moteur.get_all_normalized_forms("TACHYCARDIE_VENTRICULAIRE")
print(f"[OK] TACHYCARDIE_VENTRICULAIRE formes: {tv_forms}")
print(f"[OK] 'tv' dans formes: {'tv' in tv_forms}")

print(f"\n🚀 Moteur RAG prêt — Briques 2+3+4+5 opérationnelles (modules rechargés).")

[OK] OPENAI_API_KEY : ✅
[OK] HybridSearchEngine : 537 documents indexés
[OK] Ontologie : 292 concepts chargés
[OK] Implications : 0 règles
[OK] Barème dégressif : Gen1=90%, Gen2=80%, Gen3=70%, Gen4+=60%
[OK] TACHYCARDIE_VENTRICULAIRE formes: {'tachycardie ventriculaire', 'tv', 'vt'}
[OK] 'tv' dans formes: True

🚀 Moteur RAG prêt — Briques 2+3+4+5 opérationnelles (modules rechargés).


## 📂 Cellule 2 — Chargement des Données Réelles

- **CSV** : Réponses de 5 cardiologues sur 15 cas ECG (`ECG_Collector_Data.csv`)
- **Golden Set** : 15 dossiers avec `metadata.json` (annotations expert, diagnostic principal, poids)

Le DataFrame final (`df_flat`) contiendra une ligne par **(participant × cas)** avec le texte brut et le golden set attendu.

In [11]:
# ============================================================
# CELLULE 2 — Chargement CSV + Golden Set
# ============================================================

# ─── A) Réponses des collègues ───────────────────────────────────────────────
df_responses = pd.read_csv(EVAL_ROOT / "ECG_Collector_Data.csv")
PARTICIPANTS = df_responses['code'].tolist()
print(f"[CSV] {len(PARTICIPANTS)} participants : {PARTICIPANTS}")
print(f"[CSV] Colonnes : {list(df_responses.columns)}")

# ─── B) Golden Set (15 cas) ──────────────────────────────────────────────────
golden_cases = {}
for case_dir in sorted((EVAL_ROOT / "goldenset").iterdir()):
    meta_file = case_dir / "metadata.json"
    if case_dir.is_dir() and meta_file.exists():
        with open(meta_file, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        case_num = int(meta['name'])
        golden_cases[case_num] = {
            'case_id': meta['case_id'],
            'diagnostic_principal': meta.get('diagnostic_principal', ''),
            'annotations': meta.get('annotations', []),
            'expected_concepts': meta.get('expected_concepts', []),
        }

ALL_CASES = sorted(golden_cases.keys())
print(f"\n[GOLD] {len(golden_cases)} cas chargés :")
for num, case in sorted(golden_cases.items()):
    validants = [a['concept'] for a in case['annotations']
                 if 'validant' in a.get('annotation_role', '').lower()]
    descripteurs = [a['concept'] for a in case['annotations']
                    if 'validant' not in a.get('annotation_role', '').lower()]
    print(f"   Cas {num:2d} | {case['diagnostic_principal']:<45s} | "
          f"{len(validants)} validants, {len(descripteurs)} descripteurs")

# ─── C) Aplatir en DataFrame (1 ligne = 1 participant × 1 cas) ──────────────
rows = []
for participant in PARTICIPANTS:
    for case_num in ALL_CASES:
        golden = golden_cases[case_num]
        col_name = f'cas_{case_num:02d}'
        raw_text = df_responses.loc[df_responses['code'] == participant, col_name].values[0]

        # Extraire les concepts attendus (validants + descripteurs) avec leur rôle
        expected_ids = []
        expected_names = []
        expected_roles = []  # "validant" ou "descripteur"
        for ann in golden['annotations']:
            owl = find_owl_concept(ann['concept'])
            if owl:
                expected_ids.append(owl['ontology_id'])
                expected_names.append(ann['concept'])
                role = "validant" if "validant" in ann.get('annotation_role', '').lower() else "descripteur"
                expected_roles.append(role)

        rows.append({
            'participant': participant,
            'cas': case_num,
            'diagnostic_principal': golden['diagnostic_principal'],
            'texte_etudiant': str(raw_text).strip() if pd.notna(raw_text) else '',
            'golden_ids': expected_ids,
            'golden_names': expected_names,
            'golden_roles': expected_roles,
        })

df_flat = pd.DataFrame(rows)
df_flat['is_empty'] = df_flat['texte_etudiant'].isin(['', 'nan'])

print(f"\n[DATA] DataFrame : {len(df_flat)} lignes ({len(PARTICIPANTS)} participants × {len(ALL_CASES)} cas)")
print(f"   Textes vides : {df_flat['is_empty'].sum()}")
df_flat.head(10)

[CSV] 5 participants : ['ECG-7512', 'ECG-IDXQ', 'ECG-3RMP', 'ECG-DFLC', 'ECG-1I3Q']
[CSV] Colonnes : ['code', 'cas_01', 'cas_02', 'cas_03', 'cas_04', 'cas_05', 'cas_06', 'cas_07', 'cas_08', 'cas_09', 'cas_10', 'cas_11', 'cas_12', 'cas_13', 'cas_14', 'cas_15']

[GOLD] 15 cas chargés :
   Cas  1 | ECG normal                                    | 1 validants, 1 descripteurs
   Cas  2 | BAV complet                                   | 1 validants, 1 descripteurs
   Cas  3 | Fibrillation atriale                          | 1 validants, 1 descripteurs
   Cas  4 | Microvoltage                                  | 1 validants, 3 descripteurs
   Cas  5 | Hyperkaliémie                                 | 2 validants, 0 descripteurs
   Cas  6 | Stimulation atriale                           | 2 validants, 1 descripteurs
   Cas  7 | Bloc de branche droit complet                 | 3 validants, 0 descripteurs
   Cas  8 | Flutter droit typique                         | 1 validants, 0 descripteurs
   Cas  9 |

,participant,cas,diagnostic_principal,texte_etudiant,golden_ids,golden_names,golden_roles,is_empty
0,ECG-7512,1,ECG normal,Sinusal qrs fins \nP bifide,"[BLOC_INTERATRIAL, ECG_NORMAL]","[Bloc interatrial, ECG normal]","[descripteur, validant]",False
1,ECG-7512,2,BAV complet,bav 1 hbag bbd,"[BAV_COMPLET, ECHAPPEMENT_VENTRICULAIRE]","[BAV complet, Echappement ventriculaire]","[validant, descripteur]",False
2,ECG-7512,3,Fibrillation atriale,fibrillation atriale,"[FIBRILLATION_ATRIALE, REPOLARISATION_PRÉCOCE]","[Fibrillation atriale, Repolarisation précoce]","[validant, descripteur]",False
3,ECG-7512,4,Microvoltage,microvoltage,"[AMYLOSE, BAV_DE_TYPE_1, PERTE_DES_ONDE_Q_SEPT...","[Amylose, BAV de type 1, Perte des onde Q sept...","[descripteur, descripteur, descripteur, validant]",False
4,ECG-7512,5,Hyperkaliémie,hyperkaliemie qrs fins onde amble bav complet,"[HYPERKALIÉMIE, BAV_DE_HAUT_GRADE]","[Hyperkaliémie, BAV de haut grade]","[validant, validant]",False
5,ECG-7512,6,Stimulation atriale,stimulation atriale,"[STIMULATION_ATRIALE, BLOC_FASCICULAIRE_ANTÉRI...","[Stimulation atriale, Bloc fasciculaire antéri...","[validant, validant, descripteur]",False
6,ECG-7512,7,Bloc de branche droit complet,bbd et hbag bav 1,"[BLOC_DE_BRANCHE_DROIT_COMPLET, BAV_DE_TYPE_1,...","[Bloc de branche droit complet, BAV de type 1,...","[validant, validant, validant]",False
7,ECG-7512,8,Flutter droit typique,flutter commun qrs normaux,[FLUTTER_DROIT_TYPIQUE],[Flutter droit typique],[validant],False
8,ECG-7512,9,BAV 2 Mobitz 2,bav 2 mobitz 1,"[BAV_2_MOBITZ_2, BLOC_DE_BRANCHE_DROIT, BLOC_F...","[BAV 2 Mobitz 2, Bloc de branche droit, Bloc f...","[validant, descripteur, descripteur]",False
9,ECG-7512,10,Bloc de branche gauche complet,bloc de branche gauche,"[BLOC_DE_BRANCHE_GAUCHE_COMPLET, RYTHME_SINUSA...","[Bloc de branche gauche complet, Rythme sinusa...","[validant, descripteur, descripteur]",False


## ⚙️ Cellule 3 — Boucle de Traitement (Le Cœur du Benchmark)

Pour chaque **(participant × cas)** :
1. **Brique 2** : `extract_clinical_terms(texte)` → entités NER avec statut
2. **Brique 3** : `moteur.search_top_k(terme_brut)` → Top-K candidats
3. **Brique 4** : `resolve_term_to_ontology(terme, contexte, candidats)` → ID ontologique
4. **Scoring ensembliste** :
   - **concepts_valides_attendus** = concepts trouvés ∩ golden set → **score strict**
   - **decouvertes_additionnelles** = concepts trouvés — golden set → **valorisées mais 0 pts**
   - Score = **moyenne des % des validants attendus matchés** (découvertes hors note)

| Match | Score | Signification |
|-------|-------|---------------|
| ✅ EXACT | 100% | ID trouvé = ID attendu |
| 🟠 CHILD gen1 | 90% | Descendant direct (enfant) |
| 🟠 CHILD gen2 | 80% | Petit-enfant |
| 🟠 CHILD gen3 | 70% | Arrière-petit-enfant |
| 🔴 PARENT gen1 | 90% | Ancêtre direct (parent) |
| 🔴 PARENT gen2 | 80% | Grand-parent |
| 🔴 PARENT gen3 | 70% | Arrière-grand-parent |
| 🔵 IMPL | 100% | Auto-validé par implication |
| ❌ MISSING | 0% | Non trouvé |
| 🟢 DÉCOUVERTE | — | Concept juste mais non exigé (pas de point) |

⏱️ **Temps estimé** : ~5-10 min

In [12]:
# ============================================================
# CELLULE 3 — Pipeline complet : Extraction → RAG → Scoring
# ============================================================
import importlib
import scoring
importlib.reload(scoring)
from scoring import score_student_response, find_owl_concept, SCORE_BY_GENERATION, SCORE_GENERATION_FLOOR

def run_pipeline(texte: str, golden_names: list, golden_ids: list,
                 golden_roles: list, diagnostic_principal: str):
    """
    Pipeline RAG Neurosymbolique complet pour 1 texte étudiant.
    Score STRICT = moyenne des % des diagnostics VALIDANTS ATTENDUS uniquement.
    Les découvertes additionnelles sont trackées mais ne rapportent aucun point.
    """
    result = {
        'nb_entites': 0,
        'entites_extraites': [],
        'ids_trouves': [],
        'noms_trouves': [],
        'statuts': [],
        'methodes': [],
        'matched_expected': [],
        'missing_expected': [],
        'auto_validated': [],
        'score_final_pct': 0.0,
        'latence_s': 0.0,
        'erreur': None,
        'match_types': {},
        'partial_matches': [],
        'validant_found': 0,
        'validant_total': 0,
        'descripteur_found': 0,
        'descripteur_total': 0,
        # Nouveau : logique ensembliste
        'concepts_valides_attendus': [],
        'decouvertes_additionnelles': [],
    }
    if not texte or texte in ('', 'nan'):
        result['erreur'] = 'texte_vide'
        return result
    t0 = time.time()
    try:
        # ─── Brique 2 : Extraction NER ───────────────────────────────────
        extraction = extract_clinical_terms(texte)
        result['nb_entites'] = len(extraction.entites)

        # ─── Briques 3+4 : Recherche + Juge ─────────────────────────────
        student_matched_ids = {}
        for entite in extraction.entites:
            result['entites_extraites'].append(entite.terme_brut)
            candidats = moteur.search_top_k(entite.terme_brut)
            resolution = resolve_term_to_ontology(
                entite.terme_brut, entite.contexte_phrase, candidats
            )
            matched_id = resolution["ontology_id"]
            result['methodes'].append(resolution["method"])
            if matched_id != "NONE":
                student_matched_ids[matched_id] = entite.statut
                result['ids_trouves'].append(matched_id)
                result['noms_trouves'].append(resolution.get("concept_name", matched_id))
                result['statuts'].append(entite.statut)

        # ─── Brique 5 : Scoring ensembliste (validants attendus uniquement) ──
        scoring_result = score_student_response(
            found_ids=list(student_matched_ids.keys()),
            found_statuts=student_matched_ids,
            golden_names=golden_names,
            golden_ids=golden_ids,
            golden_roles=golden_roles,
        )

        result['matched_expected'] = scoring_result['matched_expected']
        result['missing_expected'] = scoring_result['missing_expected']
        result['auto_validated'] = scoring_result['auto_validated']
        result['score_final_pct'] = scoring_result['score_final_pct']
        result['match_types'] = scoring_result.get('match_types', {})
        result['partial_matches'] = scoring_result.get('partial_matches', [])
        result['validant_found'] = scoring_result['validant_found']
        result['validant_total'] = scoring_result['validant_total']
        result['descripteur_found'] = scoring_result['descripteur_found']
        result['descripteur_total'] = scoring_result['descripteur_total']
        # Nouveau : ensembliste
        result['concepts_valides_attendus'] = scoring_result.get('concepts_valides_attendus', [])
        result['decouvertes_additionnelles'] = scoring_result.get('decouvertes_additionnelles', [])

    except Exception as e:
        result['erreur'] = str(e)[:120]
    result['latence_s'] = round(time.time() - t0, 2)
    return result

# ─── Boucle principale ───────────────────────────────────────────────────────
print(f"🚀 Benchmark RAG Neurosymbolique : {len(df_flat)} évaluations")
print(f"   {len(PARTICIPANTS)} participants × {len(ALL_CASES)} cas")
print(f"   📊 Barème dégressif : Gen1={SCORE_BY_GENERATION[1]:.0f}%, Gen2={SCORE_BY_GENERATION[2]:.0f}%, Gen3={SCORE_BY_GENERATION[3]:.0f}%, Gen4+={SCORE_GENERATION_FLOOR:.0f}%")
print(f"{'='*90}")

all_results = []
all_match_types = []
t_start = time.time()

for idx, row in tqdm(df_flat.iterrows(), total=len(df_flat), desc="Pipeline RAG"):
    res = run_pipeline(
        texte=row['texte_etudiant'],
        golden_names=row['golden_names'],
        golden_ids=row['golden_ids'],
        golden_roles=row['golden_roles'],
        diagnostic_principal=row['diagnostic_principal'],
    )
    all_results.append(res)
    for mt in res.get('match_types', {}).values():
        all_match_types.append(mt)

elapsed = time.time() - t_start

# ─── Intégrer les résultats dans le DataFrame ────────────────────────────────
df_flat['nb_entites'] = [r['nb_entites'] for r in all_results]
df_flat['concepts_ia'] = [' | '.join(r['noms_trouves']) for r in all_results]
df_flat['entites_brutes'] = [' | '.join(r['entites_extraites']) for r in all_results]
df_flat['statuts'] = [' | '.join(r['statuts']) for r in all_results]
df_flat['methodes'] = [' | '.join(r['methodes']) for r in all_results]
df_flat['matched'] = [' | '.join(r['matched_expected']) for r in all_results]
df_flat['missing'] = [' | '.join(r['missing_expected']) for r in all_results]
df_flat['auto_validated'] = [' | '.join(r['auto_validated']) for r in all_results]
df_flat['score_final'] = [r['score_final_pct'] for r in all_results]
df_flat['latence_s'] = [r['latence_s'] for r in all_results]
df_flat['erreur'] = [r['erreur'] for r in all_results]
df_flat['validant_found'] = [r['validant_found'] for r in all_results]
df_flat['validant_total'] = [r['validant_total'] for r in all_results]
df_flat['descripteur_found'] = [r['descripteur_found'] for r in all_results]
df_flat['descripteur_total'] = [r['descripteur_total'] for r in all_results]
# Nouveau : ensembliste
df_flat['n_attendus_valides'] = [len(r['concepts_valides_attendus']) for r in all_results]
df_flat['n_decouvertes'] = [len(r['decouvertes_additionnelles']) for r in all_results]
df_flat['decouvertes_noms'] = [
    ' | '.join(d['concept_name'] for d in r['decouvertes_additionnelles'])
    for r in all_results
]

# ─── Résumé ──────────────────────────────────────────────────────────────────
df_valid = df_flat[df_flat['erreur'].isna()].copy()

print(f"\n{'='*90}")
print(f"✅ Benchmark terminé en {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Évaluations réussies : {len(df_valid)} / {len(df_flat)}")
print(f"   Score moyen (validants attendus) : {df_valid['score_final'].mean():.1f}%")
print(f"   Score médian                     : {df_valid['score_final'].median():.1f}%")
print(f"   Écart-type                       : {df_valid['score_final'].std():.1f}%")
print(f"   Latence moy/cas                  : {df_valid['latence_s'].mean():.1f}s")

# Stats méthodes
all_methodes = [m for r in all_results for m in r['methodes']]
n_cc = all_methodes.count('coupe_circuit')
n_juge = all_methodes.count('juge_llm')
n_fb = all_methodes.count('fallback_subterm')
n_none = all_methodes.count('no_candidates')
n_total = len(all_methodes)
print(f"\n   ⚡ Coupe-circuit : {n_cc}/{n_total} ({n_cc/n_total*100:.0f}%)")
print(f"   🧠 Juge LLM     : {n_juge}/{n_total} ({n_juge/n_total*100:.0f}%)")
print(f"   🔄 Fallback sub  : {n_fb}/{n_total} ({n_fb/n_total*100:.0f}%)")
print(f"   ❌ No candidates : {n_none}/{n_total} ({n_none/n_total*100:.0f}%)")

# Stats matching — maintenant avec match_types granulaires (child_gen1, parent_gen2, etc.)
n_exact = sum(1 for mt in all_match_types if mt == 'exact')
n_child_by_gen = {}
n_parent_by_gen = {}
n_impl = sum(1 for mt in all_match_types if mt == 'implication')
for mt in all_match_types:
    if mt.startswith('child_gen'):
        gen = mt.replace('child_gen', '')
        n_child_by_gen[gen] = n_child_by_gen.get(gen, 0) + 1
    elif mt.startswith('parent_gen'):
        gen = mt.replace('parent_gen', '')
        n_parent_by_gen[gen] = n_parent_by_gen.get(gen, 0) + 1

n_child_total = sum(n_child_by_gen.values())
n_parent_total = sum(n_parent_by_gen.values())

print(f"\n   🔗 Matching hiérarchique :")
print(f"      EXACT       : {n_exact}")
for gen in sorted(n_child_by_gen.keys()):
    score = SCORE_BY_GENERATION.get(int(gen), SCORE_GENERATION_FLOOR)
    print(f"      CHILD gen{gen}  : {n_child_by_gen[gen]} ({score:.0f}%)")
for gen in sorted(n_parent_by_gen.keys()):
    score = SCORE_BY_GENERATION.get(int(gen), SCORE_GENERATION_FLOOR)
    print(f"      PARENT gen{gen} : {n_parent_by_gen[gen]} ({score:.0f}%)")
print(f"      IMPLICATION : {n_impl} (auto-validé)")
print(f"      Total child : {n_child_total}, Total parent : {n_parent_total}")

# Stats ensembliste (NOUVEAU)
total_decouvertes = sum(len(r['decouvertes_additionnelles']) for r in all_results if r['erreur'] is None)
total_attendus_valides = sum(len(r['concepts_valides_attendus']) for r in all_results if r['erreur'] is None)
total_golden = sum(len(r['matched_expected']) + len(r['missing_expected']) for r in all_results if r['erreur'] is None)
print(f"\n   📊 Logique ensembliste :")
print(f"      Attendus validés (∩)     : {total_attendus_valides} / {total_golden}")
print(f"      Découvertes additionnelles : {total_decouvertes} (vrais concepts, non exigés)")
cas_avec_decouvertes = sum(1 for r in all_results if r['erreur'] is None and len(r['decouvertes_additionnelles']) > 0)
print(f"      Cas avec découvertes     : {cas_avec_decouvertes} / {len(df_valid)}")

🚀 Benchmark RAG Neurosymbolique : 75 évaluations
   5 participants × 15 cas
   📊 Barème dégressif : Gen1=90%, Gen2=80%, Gen3=70%, Gen4+=60%


Pipeline RAG: 100%|██████████| 75/75 [11:31<00:00,  9.23s/it]


✅ Benchmark terminé en 692s (11.5 min)
   Évaluations réussies : 73 / 75
   Score moyen (validants attendus) : 90.6%
   Score médian                     : 100.0%
   Écart-type                       : 22.2%
   Latence moy/cas                  : 9.5s

   ⚡ Coupe-circuit : 213/421 (51%)
   🧠 Juge LLM     : 196/421 (47%)
   🔄 Fallback sub  : 12/421 (3%)
   ❌ No candidates : 0/421 (0%)

   🔗 Matching hiérarchique :
      EXACT       : 85
      CHILD gen1  : 18 (90%)
      CHILD gen2  : 1 (80%)
      PARENT gen1 : 16 (90%)
      IMPLICATION : 0 (auto-validé)
      Total child : 19, Total parent : 16

   📊 Logique ensembliste :
      Attendus validés (∩)     : 120 / 152
      Découvertes additionnelles : 229 (vrais concepts, non exigés)
      Cas avec découvertes     : 65 / 73


## 📊 Cellule 4 — Visualisation & Audit Clinique (Dark Theme)

### A) Heatmap Participant × Cas (fond sombre)
### B) Tableau d'audit — Validant X/X + Descripteur X/X + code couleur match_type
### C) Classement des participants + Difficulté par cas
### D) Détail des cas à 0%
### E) Métriques finales
### F) 🟢 Découvertes additionnelles — Concepts justes mais non exigés par le barème

In [13]:
# ============================================================
# CELLULE 4 — Visualisation & Audit Clinique
# ============================================================
from IPython.display import display, HTML

# ═══════════════════════════════════════════════════════════════
# Palette dark-friendly
# ═══════════════════════════════════════════════════════════════
COLORS = {
    'exact':  '#4CAF50',   # Vert vif
    'child':  '#FF9800',   # Orange (pour tous child_genN)
    'parent': '#F44336',   # Rouge (pour tous parent_genN)
    'impl':   '#2196F3',   # Bleu
    'miss':   '#9E9E9E',   # Gris
    'decouverte': '#00BCD4',  # Cyan — découverte additionnelle
    'bg_dark':    '#1e1e1e',
    'bg_row':     '#2d2d2d',
    'bg_row_alt': '#252525',
    'bg_header':  '#333333',
    'text':       '#e0e0e0',
    'text_dim':   '#999999',
}

def _match_type_color(mt: str) -> str:
    """Retourne la couleur pour un match_type (child_gen1, parent_gen2, etc.)."""
    if mt == 'exact': return COLORS['exact']
    if mt.startswith('child'): return COLORS['child']
    if mt.startswith('parent'): return COLORS['parent']
    if mt == 'implication': return COLORS['impl']
    return COLORS['miss']

def _match_type_symbol(mt: str) -> str:
    """Retourne l'emoji pour un match_type."""
    if mt == 'exact': return '✅'
    if mt.startswith('child'): return '🟠'
    if mt.startswith('parent'): return '🔴'
    if mt == 'implication': return '🔵'
    return '❌'

# ═══════════════════════════════════════════════════════════════
# A) HEATMAP — Score final (Participant × Cas) — DARK THEME
# ═══════════════════════════════════════════════════════════════

def heatmap_bg(val):
    """Couleur de fond dark-friendly selon le score."""
    if pd.isna(val) or not isinstance(val, (int, float)):
        return f'background-color: {COLORS["bg_dark"]}; color: {COLORS["text_dim"]}'
    if val >= 90:  return 'background-color: #1b5e20; color: #a5d6a7; font-weight: bold'
    if val >= 70:  return 'background-color: #33691e; color: #c5e1a5'
    if val >= 50:  return 'background-color: #4e342e; color: #ffcc80'
    if val >= 20:  return 'background-color: #b71c1c; color: #ef9a9a'
    return 'background-color: #4a0000; color: #ff8a80; font-weight: bold'

pivot = df_valid.pivot_table(index='participant', columns='cas', values='score_final', aggfunc='first')
pivot['Moyenne'] = pivot.mean(axis=1).round(1)
mean_row = pivot.mean(axis=0).round(1)
mean_row.name = 'MOYENNE'
pivot = pd.concat([pivot, mean_row.to_frame().T])

print("═" * 90)
print("A) HEATMAP — Score STRICT (%) par Participant × Cas  [Attendus validés uniquement]")
print(f"   Barème dégressif : Gen1={SCORE_BY_GENERATION[1]:.0f}%, Gen2={SCORE_BY_GENERATION[2]:.0f}%, Gen3={SCORE_BY_GENERATION[3]:.0f}%, Gen4+={SCORE_GENERATION_FLOOR:.0f}%")
print("═" * 90)
display(pivot.style
    .map(heatmap_bg)
    .set_caption("🎯 Scores STRICT — Attendus validés uniquement (%) — Participant × Cas")
    .set_properties(**{'text-align': 'center', 'font-size': '12px', 'border': '1px solid #444'})
    .format(precision=1, na_rep='--')
    .set_table_styles([
        {'selector': 'th', 'props': f'background-color: {COLORS["bg_header"]}; color: {COLORS["text"]}; border: 1px solid #444;'},
        {'selector': 'caption', 'props': f'color: {COLORS["text"]}; font-size: 14px; font-weight: bold;'},
    ])
)

# ═══════════════════════════════════════════════════════════════
# B) TABLEAU D'AUDIT — Validant X/X + Descripteur X/X + Découvertes + Match Types
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("B) TABLEAU D'AUDIT — Attendus (validants & descripteurs) + Découvertes additionnelles")
print("═" * 90)

# Légende
display(HTML(f"""
<div style="background:{COLORS['bg_dark']}; padding:10px; border-radius:8px; margin-bottom:10px; font-family:monospace;">
  <b style="color:{COLORS['text']}">Légende match_type (barème dégressif par génération) :</b><br>
  <span style="color:{COLORS['exact']}; font-weight:bold"> ● EXACT (100%)</span> &nbsp;
  <span style="color:{COLORS['child']}; font-weight:bold"> ● CHILD gen1 (90%) / gen2 (80%) / gen3 (70%)</span> &nbsp;
  <span style="color:{COLORS['parent']}; font-weight:bold"> ● PARENT gen1 (90%) / gen2 (80%) / gen3 (70%)</span> &nbsp;
  <span style="color:{COLORS['impl']}; font-weight:bold"> ● IMPLICATION (auto)</span> &nbsp;
  <span style="color:{COLORS['miss']}; font-weight:bold"> ● MANQUÉ (0%)</span> &nbsp;
  <span style="color:{COLORS['decouverte']}; font-weight:bold"> ● DÉCOUVERTE (0pt, mais juste)</span>
</div>
"""))

# Construire le HTML du tableau d'audit
html_rows = []
for i, (_, row) in enumerate(df_valid.iterrows()):
    r = all_results[row.name]
    bg = COLORS['bg_row'] if i % 2 == 0 else COLORS['bg_row_alt']

    # Construire la colonne "Validants" avec code couleur
    val_parts = []
    for gn, role in zip(row['golden_names'], row['golden_roles']):
        if role != 'validant':
            continue
        mt = r['match_types'].get(gn, 'missing')
        color = _match_type_color(mt)
        symbol = _match_type_symbol(mt)
        # Afficher la génération si disponible
        gen_info = f" ({mt})" if mt not in ('exact', 'missing', 'implication') else ""
        val_parts.append(f'<span style="color:{color}" title="{mt.upper()}">{symbol} {gn}{gen_info}</span>')

    # Construire la colonne "Descripteurs" avec code couleur
    desc_parts = []
    for gn, role in zip(row['golden_names'], row['golden_roles']):
        if role != 'descripteur':
            continue
        mt = r['match_types'].get(gn, 'missing')
        color = _match_type_color(mt)
        symbol = _match_type_symbol(mt)
        desc_parts.append(f'<span style="color:{color}" title="{mt.upper()}">{symbol} {gn}</span>')

    # Construire la colonne "Découvertes additionnelles"
    dec_parts = []
    for d in r.get('decouvertes_additionnelles', []):
        dec_parts.append(
            f'<span style="color:{COLORS["decouverte"]}" title="Découverte — {d["categorie"]}">🟢 {d["concept_name"]}</span>'
        )

    vf, vt = r['validant_found'], r['validant_total']
    df_, dt = r['descripteur_found'], r['descripteur_total']
    score = r['score_final_pct']
    n_dec = len(r.get('decouvertes_additionnelles', []))

    # Couleur du score
    if score >= 90: sc = COLORS['exact']
    elif score >= 50: sc = COLORS['child']
    else: sc = COLORS['parent']

    html_rows.append(f"""
    <tr style="background:{bg}">
      <td style="color:{COLORS['text']}; padding:4px 8px; white-space:nowrap">{row['participant']}</td>
      <td style="color:{COLORS['text']}; padding:4px 8px; text-align:center">{row['cas']}</td>
      <td style="color:{COLORS['text_dim']}; padding:4px 8px; font-size:10px; max-width:180px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap">{str(row['texte_etudiant'])[:60]}</td>
      <td style="color:{COLORS['text']}; padding:4px 8px; font-weight:bold; text-align:center">{vf}/{vt}</td>
      <td style="padding:4px 8px; font-size:11px">{'<br>'.join(val_parts) if val_parts else '<span style="color:#666">—</span>'}</td>
      <td style="color:{COLORS['text']}; padding:4px 8px; font-weight:bold; text-align:center">{df_}/{dt}</td>
      <td style="padding:4px 8px; font-size:11px">{'<br>'.join(desc_parts) if desc_parts else '<span style="color:#666">—</span>'}</td>
      <td style="color:{COLORS['decouverte']}; padding:4px 8px; text-align:center; font-weight:bold">{n_dec if n_dec > 0 else '—'}</td>
      <td style="padding:4px 8px; font-size:10px">{'<br>'.join(dec_parts) if dec_parts else ''}</td>
      <td style="color:{sc}; padding:4px 8px; text-align:center; font-weight:bold; font-size:14px">{score:.0f}%</td>
    </tr>""")

audit_html = f"""
<table style="border-collapse:collapse; width:100%; font-family:monospace; font-size:12px; background:{COLORS['bg_dark']}; border:1px solid #444">
  <thead>
    <tr style="background:{COLORS['bg_header']}">
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Participant</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Cas</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Texte</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Valid.</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Détail validants</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Desc.</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Détail descripteurs</th>
      <th style="color:{COLORS['decouverte']}; padding:6px 8px; border:1px solid #444">Déc.</th>
      <th style="color:{COLORS['decouverte']}; padding:6px 8px; border:1px solid #444">Découvertes</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Score</th>
    </tr>
  </thead>
  <tbody>
    {''.join(html_rows)}
  </tbody>
</table>
"""
display(HTML(audit_html))

# ═══════════════════════════════════════════════════════════════
# C) CLASSEMENT + DIFFICULTÉ
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("C) CLASSEMENT DES PARTICIPANTS")
print("═" * 90)

p_stats = df_valid.groupby('participant').agg(
    score_moyen=('score_final', 'mean'),
    score_median=('score_final', 'median'),
    n_parfaits=('score_final', lambda x: (x >= 100).sum()),
    n_zeros=('score_final', lambda x: (x == 0).sum()),
    decouvertes_total=('n_decouvertes', 'sum'),
    latence_moy=('latence_s', 'mean'),
).round(1).sort_values('score_moyen', ascending=False)

display(p_stats.style
    .map(heatmap_bg, subset=['score_moyen', 'score_median'])
    .set_caption("🏆 Classement des participants (score strict + découvertes)")
    .set_properties(**{'text-align': 'center', 'border': '1px solid #444'})
    .set_table_styles([
        {'selector': 'th', 'props': f'background-color: {COLORS["bg_header"]}; color: {COLORS["text"]}; border: 1px solid #444;'},
        {'selector': 'td', 'props': f'background-color: {COLORS["bg_row"]}; color: {COLORS["text"]};'},
        {'selector': 'caption', 'props': f'color: {COLORS["text"]}; font-size: 14px; font-weight: bold;'},
    ])
)

print(f"\n{'═'*90}")
print("DIFFICULTÉ PAR CAS (du plus difficile au plus facile)")
print("═" * 90)

c_stats = df_valid.groupby(['cas', 'diagnostic_principal']).agg(
    score_moyen=('score_final', 'mean'),
    score_min=('score_final', 'min'),
    score_max=('score_final', 'max'),
    ecart_type=('score_final', 'std'),
    decouvertes_moy=('n_decouvertes', 'mean'),
    latence_moy=('latence_s', 'mean'),
).round(1).sort_values('score_moyen', ascending=True)

display(c_stats.style
    .map(heatmap_bg, subset=['score_moyen'])
    .set_caption("📈 Difficulté par cas ECG (+ découvertes moyennes)")
    .set_properties(**{'text-align': 'center', 'border': '1px solid #444'})
    .set_table_styles([
        {'selector': 'th', 'props': f'background-color: {COLORS["bg_header"]}; color: {COLORS["text"]}; border: 1px solid #444;'},
        {'selector': 'td', 'props': f'background-color: {COLORS["bg_row"]}; color: {COLORS["text"]};'},
        {'selector': 'caption', 'props': f'color: {COLORS["text"]}; font-size: 14px; font-weight: bold;'},
    ])
)

# ═══════════════════════════════════════════════════════════════
# D) DÉTAIL DES CAS À 0%
# ═══════════════════════════════════════════════════════════════

zeros = df_valid[df_valid['score_final'] == 0]
print(f"\n{'═'*90}")
print(f"D) DÉTAIL DES {len(zeros)} CAS À 0% (investigation)")
print("═" * 90)

for _, row in zeros.iterrows():
    r = all_results[row.name]
    print(f"\n[0%] {row['participant']} — Cas {row['cas']} — {row['diagnostic_principal']}")
    print(f"   TEXTE     : \"{row['texte_etudiant'][:120]}\"")
    print(f"   ATTENDU   : {row['golden_names']}")
    print(f"   IA trouvé : {row['concepts_ia'] or '(rien trouvé)'}")
    print(f"   MANQUANT  : {row['missing']}")
    if r.get('decouvertes_additionnelles'):
        dec_names = [d['concept_name'] for d in r['decouvertes_additionnelles']]
        print(f"   🟢 DÉCOUVERTES : {dec_names}  (justes mais non exigées → 0pt)")

# ═══════════════════════════════════════════════════════════════
# E) MÉTRIQUES FINALES
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("📊 MÉTRIQUES FINALES — Pipeline RAG Neurosymbolique (Score STRICT = Attendus uniquement)")
print(f"   Barème : EXACT=100%, CHILD/PARENT gen1=90%, gen2=80%, gen3=70%, gen4+={SCORE_GENERATION_FLOOR:.0f}%")
print("═" * 90)
print(f"   Score moyen global  : {df_valid['score_final'].mean():.1f}%")
print(f"   Score médian        : {df_valid['score_final'].median():.1f}%")
print(f"   Écart-type          : {df_valid['score_final'].std():.1f}%")
print(f"   Min / Max           : {df_valid['score_final'].min():.1f}% / {df_valid['score_final'].max():.1f}%")
print(f"   Cas parfaits (100%) : {(df_valid['score_final'] >= 100).sum()} / {len(df_valid)}")
print(f"   Cas à 0%            : {(df_valid['score_final'] == 0).sum()} / {len(df_valid)}")
print(f"   Latence totale      : {df_valid['latence_s'].sum():.0f}s ({df_valid['latence_s'].sum()/60:.1f} min)")
print(f"   Latence moy/cas     : {df_valid['latence_s'].mean():.1f}s")

# Distribution
bins = [0, 20, 40, 60, 80, 101]
labels = ['0-19%', '20-39%', '40-59%', '60-79%', '80-100%']
df_valid['tranche'] = pd.cut(df_valid['score_final'], bins=bins, labels=labels, right=False)
dist = df_valid['tranche'].value_counts().sort_index()
n = len(df_valid)
print(f"\n   Distribution des scores :")
for tranche, count in dist.items():
    bar = '█' * int(count / n * 40) + '░' * (40 - int(count / n * 40))
    print(f"      {tranche:>8s} : {bar} {count:2d} ({count/n*100:.0f}%)")

print(f"\n🎯 Verdict : Précision globale (attendus validés) = {df_valid['score_final'].mean():.1f}%")

# ═══════════════════════════════════════════════════════════════
# F) 🟢 DÉCOUVERTES ADDITIONNELLES — Analyse détaillée
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("F) 🟢 DÉCOUVERTES ADDITIONNELLES — Concepts vrais mais hors barème")
print("═" * 90)

# Collecter toutes les découvertes
all_decouvertes = []
for i, r in enumerate(all_results):
    if r['erreur'] is not None:
        continue
    row = df_flat.iloc[i]
    for d in r.get('decouvertes_additionnelles', []):
        all_decouvertes.append({
            'participant': row['participant'],
            'cas': row['cas'],
            'diagnostic': row['diagnostic_principal'],
            'concept_name': d['concept_name'],
            'categorie': d['categorie'],
            'statut': d['statut'],
        })

df_dec = pd.DataFrame(all_decouvertes) if all_decouvertes else pd.DataFrame()

if len(df_dec) > 0:
    print(f"\n   Total découvertes : {len(df_dec)} concepts")
    print(f"   Cas avec ≥1 découverte : {df_dec.groupby(['participant', 'cas']).ngroups}")

    # Top concepts découverts (les plus fréquents)
    top_dec = df_dec['concept_name'].value_counts().head(15)
    print(f"\n   Top 15 concepts découverts (non exigés mais justes) :")
    for cname, cnt in top_dec.items():
        print(f"      {cnt:3d}× — {cname}")

    # Tableau HTML des découvertes par cas
    dec_html_rows = []
    for i, (_, drow) in enumerate(df_dec.iterrows()):
        bg = COLORS['bg_row'] if i % 2 == 0 else COLORS['bg_row_alt']
        dec_html_rows.append(f"""
        <tr style="background:{bg}">
          <td style="color:{COLORS['text']}; padding:3px 6px">{drow['participant']}</td>
          <td style="color:{COLORS['text']}; padding:3px 6px; text-align:center">{drow['cas']}</td>
          <td style="color:{COLORS['text_dim']}; padding:3px 6px">{drow['diagnostic']}</td>
          <td style="color:{COLORS['decouverte']}; padding:3px 6px; font-weight:bold">🟢 {drow['concept_name']}</td>
          <td style="color:{COLORS['text_dim']}; padding:3px 6px">{drow['categorie']}</td>
          <td style="color:{COLORS['text_dim']}; padding:3px 6px">{drow['statut']}</td>
        </tr>""")

    display(HTML(f"""
    <table style="border-collapse:collapse; width:100%; font-family:monospace; font-size:11px; background:{COLORS['bg_dark']}; border:1px solid #444; margin-top:10px">
      <thead>
        <tr style="background:{COLORS['bg_header']}">
          <th style="color:{COLORS['text']}; padding:5px 6px; border:1px solid #444">Participant</th>
          <th style="color:{COLORS['text']}; padding:5px 6px; border:1px solid #444">Cas</th>
          <th style="color:{COLORS['text']}; padding:5px 6px; border:1px solid #444">Diagnostic</th>
          <th style="color:{COLORS['decouverte']}; padding:5px 6px; border:1px solid #444">Découverte</th>
          <th style="color:{COLORS['text']}; padding:5px 6px; border:1px solid #444">Catégorie</th>
          <th style="color:{COLORS['text']}; padding:5px 6px; border:1px solid #444">Statut</th>
        </tr>
      </thead>
      <tbody>
        {''.join(dec_html_rows)}
      </tbody>
    </table>
    """))

    # Insight pédagogique
    print(f"\n   💡 Insight pédagogique :")
    print(f"      Ces {len(df_dec)} découvertes montrent que les cardiologues identifient")
    print(f"      des concepts cliniquement pertinents au-delà du barème strict.")
    print(f"      Le pipeline RAG les reconnaît correctement (ontologie validée).")
    print(f"      → Le feedback LLM peut valoriser ces observations supplémentaires.")
else:
    print("\n   Aucune découverte additionnelle détectée.")

══════════════════════════════════════════════════════════════════════════════════════════
A) HEATMAP — Score STRICT (%) par Participant × Cas  [Attendus validés uniquement]
   Barème dégressif : Gen1=90%, Gen2=80%, Gen3=70%, Gen4+=60%
══════════════════════════════════════════════════════════════════════════════════════════


cas,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,Moyenne
ECG-1I3Q,100.0,100.0,90.0,100.0,45.0,95.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,88.7
ECG-3RMP,80.0,100.0,100.0,100.0,--,100.0,100.0,90.0,100.0,90.0,--,100.0,100.0,90.0,95.0,95.8
ECG-7512,90.0,0.0,100.0,100.0,95.0,50.0,96.7,100.0,0.0,90.0,100.0,100.0,100.0,100.0,95.0,81.1
ECG-DFLC,90.0,100.0,100.0,100.0,90.0,100.0,96.7,100.0,100.0,90.0,100.0,100.0,100.0,100.0,45.0,94.1
ECG-IDXQ,90.0,100.0,100.0,100.0,90.0,100.0,63.3,90.0,100.0,90.0,100.0,90.0,100.0,100.0,100.0,94.2
MOYENNE,90.0,80.0,98.0,100.0,80.0,89.0,91.3,96.0,60.0,92.0,100.0,98.0,100.0,98.0,87.0,90.8



══════════════════════════════════════════════════════════════════════════════════════════
B) TABLEAU D'AUDIT — Attendus (validants & descripteurs) + Découvertes additionnelles
══════════════════════════════════════════════════════════════════════════════════════════


Participant,Cas,Texte,Valid.,Détail validants,Desc.,Détail descripteurs,Déc.,Découvertes,Score
ECG-7512,1,Sinusal qrs fins P bifide,1/1,🟠 ECG normal (child_gen1),1/1,✅ Bloc interatrial,2,🟢 QRS fins🟢 Rythme sinusal,90%
ECG-7512,2,bav 1 hbag bbd,0/1,❌ BAV complet,0/1,❌ Echappement ventriculaire,3,🟢 BAV de type 1🟢 Bloc fasciculaire antérieur gauche🟢 Bloc de branche droit,0%
ECG-7512,3,fibrillation atriale,1/1,✅ Fibrillation atriale,0/1,❌ Repolarisation précoce,—,,100%
ECG-7512,4,microvoltage,1/1,✅ Microvoltage,1/3,🟠 Amylose❌ BAV de type 1❌ Perte des onde Q septales,—,,100%
ECG-7512,5,hyperkaliemie qrs fins onde amble bav complet,2/2,✅ Hyperkaliémie🟠 BAV de haut grade (child_gen1),0/0,—,3,🟢 BAV complet🟢 QRS fins🟢 Onde T ample,95%
ECG-7512,6,stimulation atriale,1/2,✅ Stimulation atriale❌ Bloc fasciculaire antérieur gauche,0/1,❌ Bloc intraventriculaire aspécifique,—,,50%
ECG-7512,7,bbd et hbag bav 1,3/3,🔴 Bloc de branche droit complet (parent_gen1)✅ BAV de type 1✅ Bloc fasciculaire antérieur gauche,0/0,—,1,🟢 Bloc de branche droit,97%
ECG-7512,8,flutter commun qrs normaux,1/1,✅ Flutter droit typique,0/0,—,1,🟢 QRS fins,100%
ECG-7512,9,bav 2 mobitz 1,0/1,❌ BAV 2 Mobitz 2,0/2,❌ Bloc de branche droit❌ Bloc fasciculaire postérieur gauche,2,🟢 BAV de type 2🟢 BAV 2 Mobitz 1,0%
ECG-7512,10,bloc de branche gauche,1/1,🔴 Bloc de branche gauche complet (parent_gen1),0/2,❌ Rythme sinusal❌ PR normal,1,🟢 Bloc de branche gauche,90%



══════════════════════════════════════════════════════════════════════════════════════════
C) CLASSEMENT DES PARTICIPANTS
══════════════════════════════════════════════════════════════════════════════════════════


...


══════════════════════════════════════════════════════════════════════════════════════════
DIFFICULTÉ PAR CAS (du plus difficile au plus facile)
══════════════════════════════════════════════════════════════════════════════════════════


,,score_moyen,score_min,score_max,ecart_type,decouvertes_moy,latence_moy
cas,diagnostic_principal,,,,,,
9,BAV 2 Mobitz 2,60.000000,0.000000,100.000000,54.800000,3.400000,9.300000
2,BAV complet,80.000000,0.000000,100.000000,44.700000,3.000000,7.900000
5,Hyperkaliémie,80.000000,45.000000,95.000000,23.500000,3.200000,13.300000
15,Bloc de branche gauche complet,87.000000,45.000000,100.000000,23.600000,2.000000,8.500000
6,Stimulation atriale,89.000000,50.000000,100.000000,21.900000,2.600000,11.600000
1,ECG normal,90.000000,80.000000,100.000000,7.100000,3.200000,9.500000
7,Bloc de branche droit complet,91.300000,63.300000,100.000000,15.800000,3.200000,9.500000
10,Bloc de branche gauche complet,92.000000,90.000000,100.000000,4.500000,1.800000,4.400000
8,Flutter droit typique,96.000000,90.000000,100.000000,5.500000,3.000000,8.900000



══════════════════════════════════════════════════════════════════════════════════════════
D) DÉTAIL DES 3 CAS À 0% (investigation)
══════════════════════════════════════════════════════════════════════════════════════════

[0%] ECG-7512 — Cas 2 — BAV complet
   TEXTE     : "bav 1 hbag bbd"
   ATTENDU   : ['BAV complet', 'Echappement ventriculaire']
   IA trouvé : BAV de type 1 | Bloc fasciculaire antérieur gauche | Bloc de branche droit
   MANQUANT  : BAV complet | Echappement ventriculaire
   🟢 DÉCOUVERTES : ['BAV de type 1', 'Bloc fasciculaire antérieur gauche', 'Bloc de branche droit']  (justes mais non exigées → 0pt)

[0%] ECG-7512 — Cas 9 — BAV 2 Mobitz 2
   TEXTE     : "bav 2 mobitz 1"
   ATTENDU   : ['BAV 2 Mobitz 2', 'Bloc de branche droit', 'Bloc fasciculaire postérieur gauche']
   IA trouvé : BAV de type 2 | BAV 2 Mobitz 1
   MANQUANT  : BAV 2 Mobitz 2 | Bloc de branche droit | Bloc fasciculaire postérieur gauche
   🟢 DÉCOUVERTES : ['BAV de type 2', 'BAV 2 Mobitz 1']  (just

Participant,Cas,Diagnostic,Découverte,Catégorie,Statut
ECG-7512,1,ECG normal,🟢 QRS fins,DESCRIPTEUR_ECG,present
ECG-7512,1,ECG normal,🟢 Rythme sinusal,SIGNE_ECG_PATHOLOGIQUE,present
ECG-7512,2,BAV complet,🟢 BAV de type 1,DESCRIPTEUR_ECG,present
ECG-7512,2,BAV complet,🟢 Bloc fasciculaire antérieur gauche,SIGNE_ECG_PATHOLOGIQUE,present
ECG-7512,2,BAV complet,🟢 Bloc de branche droit,SIGNE_ECG_PATHOLOGIQUE,present
ECG-7512,5,Hyperkaliémie,🟢 BAV complet,DIAGNOSTIC_URGENT,present
ECG-7512,5,Hyperkaliémie,🟢 QRS fins,DESCRIPTEUR_ECG,present
ECG-7512,5,Hyperkaliémie,🟢 Onde T ample,DESCRIPTEUR_ECG,present
ECG-7512,7,Bloc de branche droit complet,🟢 Bloc de branche droit,SIGNE_ECG_PATHOLOGIQUE,present
ECG-7512,8,Flutter droit typique,🟢 QRS fins,DESCRIPTEUR_ECG,present



   💡 Insight pédagogique :
      Ces 229 découvertes montrent que les cardiologues identifient
      des concepts cliniquement pertinents au-delà du barème strict.
      Le pipeline RAG les reconnaît correctement (ontologie validée).
      → Le feedback LLM peut valoriser ces observations supplémentaires.


In [15]:
# ============================================================
# CELLULE 5 — Analyse approfondie des résultats
# ============================================================

print("=" * 90)
print("📋 ANALYSE DÉTAILLÉE DES RÉSULTATS DU BENCHMARK")
print(f"   Barème dégressif : Gen1={SCORE_BY_GENERATION[1]:.0f}%, Gen2={SCORE_BY_GENERATION[2]:.0f}%, Gen3={SCORE_BY_GENERATION[3]:.0f}%, Gen4+={SCORE_GENERATION_FLOOR:.0f}%")
print("=" * 90)

# ─── 1. Scores par participant ────────────────────────────────────────────────
print("\n━━━ 1. SCORES PAR PARTICIPANT ━━━")
for p in p_stats.index:
    row = p_stats.loc[p]
    print(f"  {p:10s} │ moy={row['score_moyen']:5.1f}% │ med={row['score_median']:5.1f}% │ "
          f"parfaits={int(row['n_parfaits']):2d} │ zéros={int(row['n_zeros']):1d} │ "
          f"découvertes={int(row['decouvertes_total']):2d} │ latence={row['latence_moy']:.1f}s")

# ─── 2. Scores par cas (difficulté) ──────────────────────────────────────────
print("\n━━━ 2. DIFFICULTÉ PAR CAS ━━━")
for (cas, diag), row in c_stats.iterrows():
    print(f"  Cas {cas:2d} │ {diag:<55s} │ moy={row['score_moyen']:5.1f}% │ "
          f"min={row['score_min']:5.1f} │ max={row['score_max']:5.1f} │ déc={row['decouvertes_moy']:.1f}")

# ─── 3. Détail des cas à 0% ──────────────────────────────────────────────────
print(f"\n━━━ 3. INVESTIGATION DES {len(zeros)} CAS À 0% ━━━")
for _, row in zeros.iterrows():
    r = all_results[row.name]
    print(f"\n  ❌ {row['participant']} — Cas {row['cas']} — {row['diagnostic_principal']}")
    print(f"     Texte      : \"{row['texte_etudiant'][:150]}\"")
    print(f"     Attendu    : {row['golden_names']}")
    print(f"     IDs attendu: {row['golden_ids']}")
    print(f"     IA trouvé  : {r['noms_trouves']}")
    print(f"     IDs trouvés: {r['ids_trouves']}")
    print(f"     Méthodes   : {r['methodes']}")
    dec = [d['concept_name'] for d in r.get('decouvertes_additionnelles', [])]
    if dec:
        print(f"     Découvertes: {dec}")

# ─── 4. Analyse des méthodes de résolution ───────────────────────────────────
print("\n━━━ 4. MÉTHODES DE RÉSOLUTION ━━━")
print(f"  Total entités traitées : {n_total}")
print(f"  ⚡ Coupe-circuit       : {n_cc:3d} ({n_cc/n_total*100:4.1f}%) — Match exact normalisation")
print(f"  🧠 Juge LLM            : {n_juge:3d} ({n_juge/n_total*100:4.1f}%) — GPT-4o-mini QCM")
print(f"  🔄 Fallback subterm    : {n_fb:3d} ({n_fb/n_total*100:4.1f}%) — Décomposition sous-termes")
print(f"  ❌ No candidates       : {n_none:3d} ({n_none/n_total*100:4.1f}%) — Aucun candidat trouvé")

# ─── 5. Analyse du matching hiérarchique (dégressif) ─────────────────────────
print("\n━━━ 5. MATCHING HIÉRARCHIQUE (DÉGRESSIF PAR GÉNÉRATION) ━━━")
n_matched = n_exact + n_child_total + n_parent_total + n_impl
n_missing = len([g for r in all_results for g in r['missing_expected'] if r['erreur'] is None])
print(f"  Total matchés : {n_matched} / {n_matched + n_missing}")
print(f"  ✅ EXACT          : {n_exact:3d} ({n_exact/max(n_matched,1)*100:4.1f}%) — ID identique (100%)")
for gen in sorted(n_child_by_gen.keys()):
    score = SCORE_BY_GENERATION.get(int(gen), SCORE_GENERATION_FLOOR)
    print(f"  🟠 CHILD gen{gen}     : {n_child_by_gen[gen]:3d} ({n_child_by_gen[gen]/max(n_matched,1)*100:4.1f}%) — Descendant gen{gen} ({score:.0f}%)")
for gen in sorted(n_parent_by_gen.keys()):
    score = SCORE_BY_GENERATION.get(int(gen), SCORE_GENERATION_FLOOR)
    print(f"  🔴 PARENT gen{gen}    : {n_parent_by_gen[gen]:3d} ({n_parent_by_gen[gen]/max(n_matched,1)*100:4.1f}%) — Ancêtre gen{gen} ({score:.0f}%)")
print(f"  🔵 IMPLICATION     : {n_impl:3d} ({n_impl/max(n_matched,1)*100:4.1f}%) — Auto-validé")

# ─── 6. Analyse des découvertes additionnelles ───────────────────────────────
print("\n━━━ 6. DÉCOUVERTES ADDITIONNELLES ━━━")
print(f"  Total : {total_decouvertes} concepts trouvés hors barème")
print(f"  Cas concernés : {cas_avec_decouvertes} / {len(df_valid)} ({cas_avec_decouvertes/len(df_valid)*100:.0f}%)")
if len(df_dec) > 0:
    print(f"\n  Top 10 concepts les plus fréquents :")
    for cname, cnt in df_dec['concept_name'].value_counts().head(10).items():
        print(f"    {cnt:3d}× │ {cname}")
    
    print(f"\n  Répartition par catégorie :")
    for cat, cnt in df_dec['categorie'].value_counts().items():
        print(f"    {cnt:3d}× │ {cat}")

# ─── 7. Distribution des scores ──────────────────────────────────────────────
print("\n━━━ 7. DISTRIBUTION DES SCORES ━━━")
for tranche, count in dist.items():
    pct = count / n * 100
    bar = '█' * int(pct / 2.5) + '░' * (40 - int(pct / 2.5))
    print(f"  {tranche:>8s} │ {bar} │ {count:2d} cas ({pct:.0f}%)")

print(f"\n{'=' * 90}")
print(f"🎯 VERDICT GLOBAL — Barème dégressif Gen1=90% Gen2=80% Gen3=70% Gen4+=60%")
print(f"{'=' * 90}")
score_moy = df_valid['score_final'].mean()
score_med = df_valid['score_final'].median()
n_parfaits = (df_valid['score_final'] >= 100).sum()
n_zeros = (df_valid['score_final'] == 0).sum()
pct_80_100 = ((df_valid['score_final'] >= 80).sum() / len(df_valid) * 100)
print(f"  Score strict moyen : {score_moy:.1f}% (médiane {score_med:.1f}%)")
print(f"  → {pct_80_100:.0f}% des cas dans la tranche 80-100%")
print(f"  → {n_zeros} vrais échecs (cas à 0%)")
print(f"  → {total_decouvertes} découvertes additionnelles")
print(f"\n  🔧 AXES D'AMÉLIORATION :")
print(f"     1. Les {n_child_total} matchs CHILD et {n_parent_total} matchs PARENT montrent")
print(f"        que le scoring hiérarchique fonctionne (récompense la proximité)")
print(f"     2. Enrichir l'ontologie pour réduire les MISSING")
print(f"     3. Les découvertes additionnelles confirment la richesse du pipeline")

📋 ANALYSE DÉTAILLÉE DES RÉSULTATS DU BENCHMARK
   Barème dégressif : Gen1=90%, Gen2=80%, Gen3=70%, Gen4+=60%

━━━ 1. SCORES PAR PARTICIPANT ━━━
  ECG-3RMP   │ moy= 95.8% │ med=100.0% │ parfaits= 8 │ zéros=0 │ découvertes=50 │ latence=11.2s
  ECG-IDXQ   │ moy= 94.2% │ med=100.0% │ parfaits= 9 │ zéros=0 │ découvertes=74 │ latence=18.0s
  ECG-DFLC   │ moy= 94.1% │ med=100.0% │ parfaits=10 │ zéros=0 │ découvertes=35 │ latence=6.4s
  ECG-1I3Q   │ moy= 88.7% │ med=100.0% │ parfaits=11 │ zéros=1 │ découvertes=54 │ latence=8.5s
  ECG-7512   │ moy= 81.1% │ med= 96.7% │ parfaits= 7 │ zéros=2 │ découvertes=16 │ latence=3.5s

━━━ 2. DIFFICULTÉ PAR CAS ━━━
  Cas  9 │ BAV 2 Mobitz 2                                          │ moy= 60.0% │ min=  0.0 │ max=100.0 │ déc=3.4
  Cas  2 │ BAV complet                                             │ moy= 80.0% │ min=  0.0 │ max=100.0 │ déc=3.0
  Cas  5 │ Hyperkaliémie                                           │ moy= 80.0% │ min= 45.0 │ max= 95.0 │ déc=3.2
  Cas 